# Tipos de datos simples · Expresiones y representación

Representación binaria, evaluación de expresiones, tipos y conversiones. Las preguntas de "¿qué resulta?" se responden razonando primero y comprobando después con Python. Los ejercicios originales usaban Python 2; aquí se adaptan a Python 3.

## 1. Máximo con 16 bits

¿Cuál es el máximo valor representable con 16 bits en notación posicional? ¿Qué secuencia de bits le corresponde?

In [1]:
maximo = 2**16 - 1
print(maximo, format(maximo, "016b"))

65535 1111111111111111


## 2. Bits necesarios

¿Cuántos bits se necesitan para representar los números del 0 al 18?

In [2]:
print((18).bit_length())

5


## 3. Sumas en binario de 8 bits

Calcular `01111111 + 00000001`, `01010101 + 10101010` y `00000011 + 00000001` con 8 bits (si hay desbordamiento, se pierde el bit 9).

In [3]:
for a, b in (("01111111", "00000001"), ("01010101", "10101010"), ("00000011", "00000001")):
    print(f"{a} + {b} = {(int(a, 2) + int(b, 2)) % 256:08b}")

01111111 + 00000001 = 10000000
01010101 + 10101010 = 11111111
00000011 + 00000001 = 00000100


## 4. Complemento a dos

Codificar en complemento a dos de 8 bits: 4, −4, 0, 127, 1 y −1.

In [4]:
def complemento_a_dos(n, bits=8):
    # un negativo n se guarda como 2^bits + n
    return format(n % 2**bits, f"0{bits}b")

for n in (4, -4, 0, 127, 1, -1):
    print(f"{n:>4} -> {complemento_a_dos(n)}")

   4 -> 00000100
  -4 -> 11111100
   0 -> 00000000
 127 -> 01111111
   1 -> 00000001
  -1 -> 11111111


## 5. Sumas y restas en complemento a dos

Efectuar con 8 bits en complemento a dos: 4 + 4, −4 + 3, 127 − 128, 128 − 127, 1 − 1 y 1 − 2. ¿Cuáles desbordan?

In [5]:
def a_entero(bits8):
    v = int(bits8, 2)
    return v - 256 if v >= 128 else v

for a, b in ((4, 4), (-4, 3), (127, -128), (128, -127), (1, -1), (1, -2)):
    rango_ok = -128 <= a <= 127 and -128 <= b <= 127
    r = complemento_a_dos(a + b)
    print(f"{a} + {b} = {r} = {a_entero(r)}", "" if rango_ok else "(128 no cabe en 8 bits)")

4 + 4 = 00001000 = 8 
-4 + 3 = 11111111 = -1 
127 + -128 = 11111111 = -1 
128 + -127 = 00000001 = 1 (128 no cabe en 8 bits)
1 + -1 = 00000000 = 0 
1 + -2 = 11111111 = -1 


## 6. Programas en una máquina con memoria

Simular una máquina sencilla cuya memoria es una lista de celdas. Ejecutar la suma de las celdas 10 y 11 en la 12 con valores 2, −2 y 0; calcular la media de las celdas 10 a 14 y dejarla en la 15; y calcular su varianza, $\sigma^2 = \frac{1}{5}\sum (x_i - \bar{x})^2$, usando solo operaciones entre celdas.

In [6]:
memoria = [0] * 20

def sumar(a, b, c):
    memoria[c] = memoria[a] + memoria[b]

def restar(a, b, c):
    memoria[c] = memoria[a] - memoria[b]

def multiplicar(a, b, c):
    memoria[c] = memoria[a] * memoria[b]

def dividir(a, b, c):
    memoria[c] = memoria[a] / memoria[b]

memoria[10:13] = [2, -2, 0]
sumar(10, 11, 12)
print("suma:", memoria[12])

memoria[10:15] = [3, 5, 8, 2, 7]
memoria[16], memoria[17] = 0, 5
for i in range(10, 15):
    sumar(16, i, 16)
dividir(16, 17, 15)
print("media:", memoria[15])

memoria[18] = 0
for i in range(10, 15):
    restar(i, 15, 19)
    multiplicar(19, 19, 19)
    sumar(18, 19, 18)
dividir(18, 17, 15)
print("varianza:", memoria[15])

suma: 0
media: 5.0
varianza: 5.2


## 7. Algoritmos sencillos

Diseñar algoritmos para el área de un círculo dado su radio y para el IVA (16 %) de un producto dado su precio sin IVA. ¿Es un algoritmo escribir todos los decimales de $\pi$?

No: un algoritmo debe terminar en un número finito de pasos, y $\pi$ tiene infinitos decimales.

In [7]:
from math import pi

def area_circulo(r):
    return pi * r**2

def iva(precio, tipo=16):
    return precio * tipo / 100

print(area_circulo(2), iva(100))

12.566370614359172 16.0


## 8. Árboles sintácticos

Escribir con el mínimo de paréntesis las expresiones de los árboles `((1 - 2) + 3) + 4`, `1 + (2 - (3 + 4))` y `(1 - 2) + (3 + 4)`, y comprobar cómo agrupa Python `1 + 2 + 3 + 4`, `1 - 2 - 3 - 4` y `1 - (2 - (3 - 4) + 1)`.

In [8]:
import ast

print(1 - 2 + 3 + 4, 1 + (2 - (3 + 4)), 1 - 2 + (3 + 4))
for expr in ("1 + 2 + 3 + 4", "1 - 2 - 3 - 4", "1 - (2 - (3 - 4) + 1)"):
    # los operadores binarios se asocian por la izquierda
    print(expr, "=", eval(expr), "|", ast.dump(ast.parse(expr, mode="eval").body)[:70], "...")

6 -4 6
1 + 2 + 3 + 4 = 10 | BinOp(left=BinOp(left=BinOp(left=Constant(value=1), op=Add(), right=Co ...
1 - 2 - 3 - 4 = -8 | BinOp(left=BinOp(left=BinOp(left=Constant(value=1), op=Sub(), right=Co ...
1 - (2 - (3 - 4) + 1) = -3 | BinOp(left=Constant(value=1), op=Sub(), right=BinOp(left=BinOp(left=Co ...


## 9. Evaluar expresiones

Evaluar `2 + 3 + 1 + 2`, `2 + 3 * 1 + 2`, `(2 + 3) * 1 + 2`, `(2 + 3) * (1 + 2)`, `+---6` y `-+-+6`.

In [9]:
for expr in ("2 + 3 + 1 + 2", "2 + 3 * 1 + 2", "(2 + 3) * 1 + 2", "(2 + 3) * (1 + 2)", "+---6", "-+-+6"):
    print(f"{expr:20} = {eval(expr)}")

2 + 3 + 1 + 2        = 8
2 + 3 * 1 + 2        = 7
(2 + 3) * 1 + 2      = 7
(2 + 3) * (1 + 2)    = 15
+---6                = -6
-+-+6                = 6


## 10. Traducir expresiones matemáticas

Traducir con el mínimo de paréntesis: $2 + 3\cdot\frac{6}{2}$, $\frac{4+6}{2+3}$, $(4/2)^5$, $(4/2)^{5+1}$, $(-3)^2$ y $-(3^2)$. Resultados esperados: 11, 2, 32, 64, 9 y −9.

In [10]:
print(2 + 3 * 6 / 2, (4 + 6) / (2 + 3), (4 / 2) ** 5, (4 / 2) ** (5 + 1), (-3) ** 2, -3**2)

11.0 2.0 32.0 64.0 9 -9


## 11. Tipos en las operaciones

¿Qué resulta de `1 / 2 / 4.0`, `1 / 2.0 / 4.0`, `1 / 2.0 / 4`, `1.0 / 2 / 4`, `4 ** .5`, `4.0 ** (1 / 2)`, `4.0 ** (1 / 2) + 1 / 2`, `4.0 ** (1.0 / 2) + 1 / 2.0`, `3e3 / 10`, `10 / 5e-3`, `10 / 5e-3 + 1` y `3 / 2 + 1`? En Python 3 `/` siempre da `float` (en Python 2, `1 / 2` valía 0).

In [11]:
exprs = ["1 / 2 / 4.0", "1 / 2.0 / 4.0", "1 / 2.0 / 4", "1.0 / 2 / 4", "4 ** .5", "4.0 ** (1 / 2)",
         "4.0 ** (1 / 2) + 1 / 2", "4.0 ** (1.0 / 2) + 1 / 2.0", "3e3 / 10", "10 / 5e-3", "10 / 5e-3 + 1", "3 / 2 + 1"]
for e in exprs:
    print(f"{e:28} = {eval(e)!r}")

1 / 2 / 4.0                  = 0.125
1 / 2.0 / 4.0                = 0.125
1 / 2.0 / 4                  = 0.125
1.0 / 2 / 4                  = 0.125
4 ** .5                      = 2.0
4.0 ** (1 / 2)               = 2.0
4.0 ** (1 / 2) + 1 / 2       = 2.5
4.0 ** (1.0 / 2) + 1 / 2.0   = 2.5
3e3 / 10                     = 300.0
10 / 5e-3                    = 2000.0
10 / 5e-3 + 1                = 2001.0
3 / 2 + 1                    = 2.5


## 12. Comparaciones encadenadas

¿Qué muestran `True == True != False`, `1 < 2 < 3 < 4 < 5`, `(1 < 2 < 3) and (4 < 5)`, `1 < 2 < 4 < 3 < 5` y `(1 < 2 < 4) and (3 < 5)`?

In [12]:
print(True == True != False, 1 < 2 < 3 < 4 < 5, (1 < 2 < 3) and (4 < 5), 1 < 2 < 4 < 3 < 5, (1 < 2 < 4) and (3 < 5))

True True True False True


## 13. Identificadores válidos

¿Son válidos `Identificador`, `Indice\dos`, `Dos palabras`, `__`, `12horas`, `hora12`, `desviación`, `año`, `from`, `var!`, `'var'`, `import_from`, `UnaVariable`, `a(b)`, `12`, `uno.dos`, `x`, `área`, `area-rect`, `x_______1`, `________1`, `_x_` y `x_x`? Python 3 sí admite letras acentuadas y `ñ`.

In [13]:
import keyword

nombres = ["Identificador", "Indice\\dos", "Dos palabras", "__", "12horas", "hora12", "desviación", "año", "from",
           "var!", "'var'", "import_from", "UnaVariable", "a(b)", "12", "uno.dos", "x", "área", "area-rect",
           "x_______1", "________1", "_x_", "x_x"]
for n in nombres:
    print(f"{n:15} {n.isidentifier() and not keyword.iskeyword(n)}")

Identificador   True
Indice\dos      False
Dos palabras    False
__              True
12horas         False
hora12          True
desviación      True
año             True
from            False
var!            False
'var'           False
import_from     True
UnaVariable     True
a(b)            False
12              False
uno.dos         False
x               True
área            True
area-rect       False
x_______1       True
________1       True
_x_             True
x_x             True


## 14. Asignaciones

¿Qué resulta de `x = 10; x = x * 10; x`? ¿Y de la secuencia `z = 2`, `z += 2`, `z += 2 - 2`, `z *= 2`, `z *= 1 + 1`, `z /= 2`, `z %= 3`, `z /= 3 - 1`, `z -= 2 + 1`, `z -= 2`, `z **= 3`?

In [14]:
x = 10
x = x * 10
print(x)
z = 2
for op in ("z += 2", "z += 2 - 2", "z *= 2", "z *= 1 + 1", "z /= 2", "z %= 3", "z /= 3 - 1", "z -= 2 + 1", "z -= 2", "z **= 3"):
    exec(op)
    print(f"{op:12} -> {z}")

100
z += 2       -> 4
z += 2 - 2   -> 4
z *= 2       -> 8
z *= 1 + 1   -> 16
z /= 2       -> 8.0
z %= 3       -> 2.0
z /= 3 - 1   -> 1.0
z -= 2 + 1   -> -2.0
z -= 2       -> -4.0
z **= 3      -> -64.0


## 15. Polinomios

Evaluar $x^4 + x^3 + 2x^2 - x$ en $x = 1.1$ (≈ 4.1151) y $x^4 + x^3 + \frac{1}{2}x^2 - x$ en $x = 10$ como `float` (11040.0).

In [15]:
x = 1.1
print(round(x**4 + x**3 + 2 * x**2 - x, 4))
x = 10
print(x**4 + x**3 + x**2 / 2 - x)

4.1151
11040.0


## 16. Operaciones con cadenas

Evaluar en orden `a = 'b'`, `a + 'b'`, `a + 'a'`, `a * 2 + 'b' * 3` y `2 * (a + 'b')`; y además `'a' * 3 + '/*' * 5 + 2 * 'abc' + '+'`, `(4 * '<' + 'abcba' + '>' * 4) * 2`, `'10' * 5 + 4 * ('=' + '-' * 3 + '=')` y `2 * '12' + '.' + '3' * 3 + 'e-' + 4 * '76'`.

In [16]:
a = "b"
print(a + "b", a + "a", a * 2 + "b" * 3, 2 * (a + "b"))
print("a" * 3 + "/*" * 5 + 2 * "abc" + "+")
palindromo = "abcba"
print((4 * "<" + palindromo + ">" * 4) * 2)
subcadena = "=" + "-" * 3 + "="
print("10" * 5 + 4 * subcadena)
print(2 * "12" + "." + "3" * 3 + "e-" + 4 * "76")

bb ba bbbbb bbbb
aaa/*/*/*/*/*abcabc+
<<<<abcba>>>><<<<abcba>>>>
1010101010=---==---==---==---=
1212.333e-76767676


## 17. Construir cadenas con regularidades

Generar con concatenación y repetición `'%%%%%./././<-><->'`, `'(@)(@)(@)======(@)(@)(@)======'`, `'asdfasdfasdf=-=-=-=-=-=-=-??????asdfasdf'` y `'........*****---*****---........*****---*****---'`.

In [17]:
print("%" * 5 + "./" * 3 + "<->" * 2)
print(("(@)" * 3 + "=" * 6) * 2)
print("asdf" * 3 + "=-" * 7 + "?" * 6 + "asdf" * 2)
print(("." * 8 + ("*" * 5 + "-" * 3) * 2) * 2)

%%%%%./././<-><->
(@)(@)(@)======(@)(@)(@)======
asdfasdfasdf=-=-=-=-=-=-=-??????asdfasdf
........*****---*****---........*****---*****---


## 18. Comparar cadenas

¿Qué dan `'abalorio' < 'abecedario'`, `'abecedario' < 'abecedario'`, `'abecedario' <= 'abecedario'`, `'Abecedario' < 'abecedario'`, `'Abecedario' == 'abecedario'`, `124 < 13`, `'124' < '13'` y `' a' < 'a'`? Las cadenas se comparan carácter a carácter por su código.

In [18]:
print("abalorio" < "abecedario", "abecedario" < "abecedario", "abecedario" <= "abecedario")
print("Abecedario" < "abecedario", "Abecedario" == "abecedario", 124 < 13, "124" < "13", " a" < "a")
print(ord("A"), ord("a"), ord(" "))

True False True
True False False True True
65 97 32


## 19. Funciones predefinidas

Calcular en una expresión el valor absoluto del redondeo de −3.2; convertir a cadena 5011/10000 redondeado a 3 decimales; evaluar `str(2.1) + str(1.2)`, `int(str(2) + str(3))`, `str(int(12.3)) + '0'`, `int('2' + '3')`, `str(2 + 3)` y `str(int(2.1) + float(3))`.

In [19]:
print(abs(round(-3.2)), str(round(5011 / 10000, 3)))
print(str(2.1) + str(1.2), int(str(2) + str(3)), str(int(12.3)) + "0", int("2" + "3"), str(2 + 3), str(int(2.1) + float(3)))

3 0.501
2.11.2 23 120 23 5 5.0


## 20. Funciones matemáticas

Evaluar `int(exp(2 * log(3)))`, `round(4 * sin(3 * pi / 2))`, `abs(log10(.01) * sqrt(25))` y `round(3.21123 * log10(1000), 3)`.

In [20]:
from math import exp, log, log10, pi, sin, sqrt

# exp(2·log 3) = 9 con error de redondeo: int() trunca
print(exp(2 * log(3)), int(exp(2 * log(3))), round(4 * sin(3 * pi / 2)), abs(log10(.01) * sqrt(25)), round(3.21123 * log10(1000), 3))

9.000000000000002 9 -4 10.0 9.634


## 21. Áreas y perímetros con datos fijos y leídos

Cuadrado de lado 3 (perímetro 12, área 9), triángulo de base 3 y altura 5 (7.5), rectángulo 4 × 6 (perímetro 20, área 24); después, pidiendo los datos al usuario.

In [21]:
print(4 * 3, 3**2, 3 * 5 / 2, 2 * (4 + 6), 4 * 6)
lado = float(input("Lado del cuadrado: "))
print("Perímetro:", round(4 * lado, 2), "Área:", round(lado**2, 2))
a, b = float(input("Lado 1: ")), float(input("Lado 2: "))
print("Perímetro:", 2 * (a + b), "Área:", a * b)
base, altura = float(input("Base: ")), float(input("Altura: "))
print("Área:", base * altura / 2)

12 9 7.5 20 24
Lado del cuadrado: 1.1
Perímetro: 4.4 Área: 1.21
Lado 1: 1
Lado 2: 5
Perímetro: 12.0 Área: 5.0
Base: 10
Altura: 100
Área: 500.0


## 22. Nombre repetido mil veces

Pedir un nombre y mostrarlo 1000 veces separado por espacios usando concatenación y repetición.

In [22]:
nombre = input("Nombre: ")
texto = (nombre + " ") * 1000
print(texto[:60], "...", len(texto.split()))

Nombre: Ana
Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana Ana  ... 1000


## 23. Formato con `%`

¿Qué muestran `'%d' % 1`, `'%d %d' % (1, 2)`, `'%d%d' % (1, 2)`, `'%d, %d' % (1, 2)`, `print(1, 2)` y `'%d 2' % 1`? ¿Por qué `'número %d y número %d'.upper() % (1, 2)` da error? ¿Cómo se reserva espacio para una cadena de 10 casillas?

`upper()` se aplica antes que `%` y convierte `%d` en `%D`, que no es una marca válida.

In [23]:
print("%d" % 1)
print("%d %d" % (1, 2))
print("%d%d" % (1, 2))
print("%d, %d" % (1, 2))
print(1, 2)
print("%d 2" % 1)
print(("número %d y número %d" % (1, 2)).upper())
try:
    print("número %d y número %d".upper() % (1, 2))
except ValueError as e:
    print("ValueError:", e)
print("[%10s] [%-10s]" % ("hola", "hola"))

1
1 2
12
1, 2
1 2
1 2
NÚMERO 1 Y NÚMERO 2
ValueError: unsupported format character 'D' (0x44) at index 8
[      hola] [hola      ]


## 24. `print` con comas

¿Qué diferencia hay entre `print('Hola,', nombre, '.')` y `print('Hola, ' + nombre + '.')`? `print` separa sus argumentos con un espacio, así que el primero deja un espacio antes del punto.

In [24]:
nombre = "Ana"
print("Hola,", nombre, ".")
print("Hola, " + nombre + ".")
print("Hola,", nombre, ".", sep=" ")
print("Hola, ", nombre, ".", sep="")

Hola, Ana .
Hola, Ana.
Hola, Ana .
Hola, Ana.
